In [2]:
import plotly.io as pio
print(pio.renderers)
pio.renderers.default = "browser"

Renderers configuration
-----------------------
    Default renderer: 'plotly_mimetype'
    Available renderers:
        ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
         'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
         'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
         'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
         'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']



In [3]:
import plotly.offline as pyo
import numpy as np
import plotly.graph_objects as go
from scipy.stats import landau
import ipywidgets as widgets
from IPython.display import display
from operator import add

In [4]:
# Define the figure

x = np.linspace(-5, 20, 1000)
y = landau.pdf(x)

# Incoming signal
sig = go.Scatter(x=x, y=y, name='sig')

# Delayed signal
delay = 1
delsig_y = np.interp(x - delay, x, y, left=0, right=0)
delsig = go.Scatter(x=x, y=delsig_y, name='delsig')

# Attenuated and flipped signal
att = 0.2
attsig_y = -att * y
attsig = go.Scatter(x=x, y=attsig_y, name='attsig')

# CFD trigger sig (addition of delsig and attsig)
CFDsig_y = delsig_y + attsig_y
CFDsig = go.Scatter(x=x, y=CFDsig_y, name='CFDsig')

fig = go.FigureWidget(
    data=[sig, delsig, attsig, CFDsig],
    layout=go.Layout(title="CFD Simulation", xaxis_range=(-5, 10), yaxis_range=(-0.7,0.7), width=600, height=400)
)

# Create sliders
loc_slider = widgets.FloatSlider(min=-2, max=5, step=0.1, value=0, description='incident time')
scale_slider = widgets.FloatSlider(min=0.5, max=2, step=0.1, value=1, description='scale')
delay_slider = widgets.FloatSlider(min=0.5, max=6, step=0.1, value=1, description='delay')
att_slider = widgets.FloatSlider(min=0.05, max=1, step=0.05, value=0.2, description='attenuation')

# Figure update function
def update(change=None):
    delay = delay_slider.value
    att = att_slider.value
    loc = loc_slider.value
    scale = scale_slider.value
    
    y = landau.pdf(x, loc=loc, scale=scale)
    
    # delsig_x = x + delay
    
    delsig_y = np.interp(x - delay, x, y, left=0, right=0)
    attsig_y = -att * y
    
    CFDsig_y = delsig_y + attsig_y

    with fig.batch_update():
        fig.update_traces(y=y, selector={'name': 'sig'})
        fig.update_traces(y=delsig_y, selector={'name': 'delsig'})
        fig.update_traces(y=attsig_y, selector={'name': 'attsig'})
        fig.update_traces(y=CFDsig_y, selector={'name': 'CFDsig'})

# Figure update upon "observation" of slider change
loc_slider.observe(update, names='value')
scale_slider.observe(update, names='value')
delay_slider.observe(update, names='value')
att_slider.observe(update, names='value')

slider_box = widgets.VBox([loc_slider, scale_slider, delay_slider, att_slider])
layout = widgets.Layout(align_items='center')
display(widgets.HBox([fig, slider_box], layout=layout))

    'data': [{'name': 'sig',
              'type': 'scatter',
              'uid…

In [82]:
# 1. Generate Landau PDF data
x = np.linspace(-5, 20, 500)
# landau.pdf() calculates the Landau PDF
pdf = landau.pdf(x, loc=1, scale=10)

# Create the Plotly figure
fig = go.Figure()

# Add the PDF line
fig.add_trace(go.Scatter(
    x=x, 
    y=pdf, 
    mode='lines', 
    name='Landau PDF',
    line=dict(color='firebrick', width=2)
))

# Update layout for better appearance
fig.update_layout(
    title='Landau Distribution Probability Density Function',
    xaxis_title='Standardized Energy Loss',
    yaxis_title='Probability Density',
    template='plotly_white'
)

fig.show()


In [47]:
# Generate Landau PDF data
x = np.linspace(-5, 20, 500)
loc_vals = np.linspace(-10, 10, 10)
scale_vals = np.linspace(0.5, 5, 10)

# Add trace to figure for every possible dist. with combinations of parameters
fig = go.Figure()

for l in loc_vals:
    for s in scale_vals:
        fig.add_trace(
            go.Scatter(
                x=x,
                y=landau.pdf(x, loc=l, scale = s),
                visible=False,
                name=f"l={l:.2f}\n s={s:.2f}",
                line=dict(color='firebrick', width=2)
            )
        )
    
fig.data[0].visible = True

# Create sliding bars for parameters
# Array for indexing sliding bar positions
sliBars = []

# loc parameter sliding bar
steps_loc = []
n_scale = len(scale_vals)

# for i in range(len(loc_vals):
#     step = dict(
#         method="update",
#         args=[{"visible": [j == i for j in range(len(loc_vals))]}],
#         label=str(i)
#     )
#     steps_loc.append(step)
               
for i, loc in enumerate(loc_vals):
    visible = [False]*len(fig.data)

    for j in range(n_scale):
        visible[i*n_scale + j] = True

    step = dict(
        method="update",
        args=[{"visible": visible}],
        label=f"{loc:.2f}"
    )
    steps_loc.append(step)
        
steps_scale = []
# for i in range(len(scale_vals)):
#     step = dict(
#         method="update",
#         args=[{"visible": [j == i for j in range(scale_len)]}],
#         label=str(i)
#     )
#     steps_scale.append(step)

n_loc = len(loc_vals)

for j, scale in enumerate(scale_vals):
    visible = [False]*len(fig.data)

    for i in range(n_loc):
        visible[i*n_scale + j] = True

    step = dict(
        method="update",
        args=[{"visible": visible}],
        label=f"{scale:.2f}"
    )

    steps_scale.append(step)

fig.update_layout(
    sliders=[dict(active=0, steps=steps_loc), dict(active=0, steps=steps_scale, y=-0.15)]
)

fig.show()

In [5]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import FloatSlider, VBox, interactive
from IPython.display import display

x = np.linspace(0, 10, 100)

def plot(y_shift):
    fig = go.Figure(data=[go.Scatter(x=x, y=np.sin(x) + y_shift)])
    fig.show()

slider = FloatSlider(min=-5, max=5, step=0.1, value=0)
interactive_plot = interactive(plot, y_shift=slider)
display(interactive_plot)

interactive(children=(FloatSlider(value=0.0, description='y_shift', max=5.0, min=-5.0), Output()), _dom_classe…

In [16]:
int_range = widgets.IntSlider()
output2 = widgets.Output()

display(int_range, output2)

def on_value_change(change):
    with output2:
        print(change['new'])
        print(change)

int_range.observe(on_value_change, names='value')

IntSlider(value=0)

Output()

In [7]:
import sys
print(sys.executable)

/home/psi-hep-micro/Desktop/CHENG/Constant-Fraction-Discrimination/venv/bin/python3


In [1]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)